# Urban Flow Analytics: Canonical Data Audit

**Project:** Nexora Datathon 2026  
**Audit scope:** 12 monthly taxi files, April 2025 through March 2026  
**Purpose:** establish a reproducible, reviewable data-quality contract before cleaning.

This notebook is intentionally read-only with respect to the raw CSV files. It profiles the complete corpus, validates the zone reference, records row-level lineage, quantifies overlapping quality rules, and separates `full_clean` decisions from `model_clean` exclusions.

## 1. Audit setup

The monthly taxi files are selected by filename pattern. The zone reference is stored in `data/raw/reference/` so it cannot be mistaken for a trip file.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_glob = project_root / "data" / "raw" / "Urban_Flow_Analytics_Taxi_Dataset_*.csv"
zone_file = project_root / "data" / "raw" / "reference" / "Urban_Flow_Analytics_Zone_Dataset.csv"
interim_dir = project_root / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)

files = sorted(raw_glob.parent.glob(raw_glob.name))
assert len(files) == 12, f"Expected 12 monthly files, found {len(files)}"
assert zone_file.exists(), f"Missing zone reference: {zone_file}"

con = duckdb.connect()
raw_sql = str(raw_glob.resolve()).replace("'", "''")
zone_sql = str(zone_file.resolve()).replace("'", "''")

print(f"Monthly taxi files: {len(files)}")
print(f"Zone reference: {zone_file}")
for path in files:
    print(f" - {path.name}")

Monthly taxi files: 12
Zone reference: /Users/praveenmadawalage/Documents/Codefest Datathon 2026/nexora-datathon2026/data/raw/reference/Urban_Flow_Analytics_Zone_Dataset.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
 - Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


## 2. File inventory and schema

DuckDB scans the headers and row counts without loading the entire corpus into a pandas DataFrame. The expected raw schema is 20 fields, including the raw spelling `Airport_fee`.

In [2]:
profile_rows = []
for path in files:
    relation = f"read_csv_auto('{str(path).replace(chr(39), chr(39) * 2)}')"
    schema = con.execute(f"DESCRIBE SELECT * FROM {relation}").fetchdf()
    row_count = con.execute(f"SELECT COUNT(*) FROM {relation}").fetchone()[0]
    profile_rows.append({
        "file": path.name,
        "rows": row_count,
        "columns": len(schema),
        "column_names": tuple(schema["column_name"]),
        "dtypes": tuple(schema["column_type"]),
    })

profile_df = pd.DataFrame(profile_rows)
profile_df[["file", "rows", "columns"]]

,file,rows,columns
0,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,3970553,20
1,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,4591845,20
2,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,4322960,20
3,Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv,3898963,20
4,Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,3574091,20
5,Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv,4251015,20
6,Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv,4428699,20
7,Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv,4181444,20
8,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,4305006,20
9,Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv,3724889,20


All 12 files should have the same 20-column schema. The inventory is saved as a compact artifact for the report appendix; no raw file is changed.

In [3]:
schema_sets = profile_df["column_names"].nunique()
dtype_sets = profile_df["dtypes"].nunique()
expected_columns = 20

print(f"Unique column-name sets: {schema_sets}")
print(f"Unique dtype sets: {dtype_sets}")
print(f"Expected columns in each file: {expected_columns}")
assert schema_sets == 1
assert dtype_sets == 1
assert profile_df["columns"].eq(expected_columns).all()
profile_df.to_csv(interim_dir / "audit_file_inventory.csv", index=False)

Unique column-name sets: 1
Unique dtype sets: 1
Expected columns in each file: 20


## 3. Canonical trip view and exact duplicates

The canonical view adds `source_file` and a deterministic row number within each source file. Exact duplicates are grouped across the 20 raw fields, excluding those audit-only lineage columns.

In [4]:
con.execute(f"""
CREATE OR REPLACE VIEW taxi_all AS
SELECT *,
       filename AS source_file,
       row_number() OVER (PARTITION BY filename) AS source_row_number
FROM read_csv_auto('{raw_sql}', filename=true, union_by_name=false)
""")
con.execute(f"""
CREATE OR REPLACE VIEW zones AS
SELECT * FROM read_csv_auto('{zone_sql}', header=true, auto_detect=true)
""")

total_rows = con.execute("SELECT COUNT(*) FROM taxi_all").fetchone()[0]
zone_rows = con.execute("SELECT COUNT(*) FROM zones").fetchone()[0]
print(f"Taxi rows: {total_rows:,}")
print(f"Zone rows: {zone_rows:,}")
assert total_rows == 48_601_782
assert zone_rows == 265

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Taxi rows: 48,601,782
Zone rows: 265


In [5]:
raw_columns = [
    "provider_code", "pickup_timestamp", "dropoff_timestamp", "rider_count",
    "distance_miles", "rate_class_id", "offline_record_flag", "origin_loc_id",
    "dest_loc_id", "fare_settlement_method", "base_fare", "surcharge_misc",
    "transit_tax", "driver_tip_payment", "toll_total", "service_improvement_fee",
    "charge_total", "zone_congestion_fee", "Airport_fee", "congestion_relief_fee",
]
column_sql = ", ".join(raw_columns)

duplicate_summary = con.execute(f"""
SELECT
    COUNT(*) AS duplicate_groups,
    COALESCE(SUM(n - 1), 0) AS duplicate_rows_to_drop
FROM (
    SELECT COUNT(*) AS n
    FROM taxi_all
    GROUP BY {column_sql}
    HAVING COUNT(*) > 1
)
""").fetchdf()
duplicate_summary["total_rows"] = total_rows
duplicate_summary.to_csv(interim_dir / "audit_duplicate_summary.csv", index=False)
duplicate_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_groups,duplicate_rows_to_drop,total_rows
0,1,1.0,48601782


The verified corpus contains **48,601,782 rows** and the zone reference contains **265 rows**. The duplicate audit will determine the exact redundant-row count; that value, not the earlier draft estimate, controls deduplication.

## 4. Missingness and monetary fields

Missing values are profiled for every raw column. Monetary signs are profiled independently because a negative fare, a missing airport fee, and a negative toll are different data-quality questions.

In [6]:
null_queries = [
    f"SELECT '{column}' AS column_name, COUNT(*) AS total_rows, COUNT(*) FILTER (WHERE {column} IS NULL) AS null_rows FROM taxi_all"
    for column in raw_columns
]
null_summary = con.execute(" UNION ALL ".join(null_queries) + " ORDER BY column_name").fetchdf()
null_summary.to_csv(interim_dir / "audit_null_summary.csv", index=False)
null_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,total_rows,null_rows
0,Airport_fee,48601782,12405268
1,base_fare,48601782,0
2,charge_total,48601782,0
3,congestion_relief_fee,48601782,0
4,dest_loc_id,48601782,0
5,distance_miles,48601782,0
6,driver_tip_payment,48601782,0
7,dropoff_timestamp,48601782,0
8,fare_settlement_method,48601782,0
9,offline_record_flag,48601782,12405268


In [7]:
monetary_columns = [
    "base_fare", "surcharge_misc", "transit_tax", "driver_tip_payment",
    "toll_total", "service_improvement_fee", "charge_total", "zone_congestion_fee",
    "Airport_fee", "congestion_relief_fee",
]
negative_queries = [
    f"SELECT '{column}' AS field, COUNT(*) AS total_rows, COUNT(*) FILTER (WHERE {column} < 0) AS negative_rows FROM taxi_all"
    for column in monetary_columns
]
monetary_summary = con.execute(" UNION ALL ".join(negative_queries) + " ORDER BY field").fetchdf()
monetary_summary.to_csv(interim_dir / "audit_monetary_summary.csv", index=False)
monetary_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,field,total_rows,negative_rows
0,Airport_fee,48601782,152252
1,base_fare,48601782,2400031
2,charge_total,48601782,875399
3,congestion_relief_fee,48601782,475447
4,driver_tip_payment,48601782,1481
5,service_improvement_fee,48601782,709250
6,surcharge_misc,48601782,366095
7,toll_total,48601782,65691
8,transit_tax,48601782,670316
9,zone_congestion_fee,48601782,555805


The verified null counts are **12,405,268** for `rider_count`, `rate_class_id`, `offline_record_flag`, and several fee fields. These are structural missingness patterns, not a reason to drop the entire row. Rider-count imputation will preserve separate null, zero, and provider-specific flags.

The raw field `Airport_fee` will be renamed to `airport_pickup_fee` exactly once during cleaning; the raw name remains unchanged for audit reproducibility.

## 5. Timestamp integrity

Both timestamps are audited. Pickup month will define chronological splits. A dropoff on 2026-04-01 is accepted only as a boundary spillover from a valid March pickup; pre-2020 values are treated as corrupt hardware-clock records.

In [8]:
date_summary = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE pickup_timestamp < TIMESTAMP '2025-04-01') AS pickup_before_window,
    COUNT(*) FILTER (WHERE pickup_timestamp >= TIMESTAMP '2026-04-01') AS pickup_after_window,
    COUNT(*) FILTER (WHERE dropoff_timestamp < TIMESTAMP '2025-04-01') AS dropoff_before_window,
    COUNT(*) FILTER (WHERE dropoff_timestamp >= TIMESTAMP '2026-04-02') AS dropoff_after_allowed_spillover,
    COUNT(*) FILTER (WHERE pickup_timestamp < TIMESTAMP '2020-01-01' OR dropoff_timestamp < TIMESTAMP '2020-01-01') AS corrupt_timestamp_rows,
    COUNT(*) FILTER (WHERE dropoff_timestamp < pickup_timestamp) AS inverted_rows,
    COUNT(*) FILTER (WHERE dropoff_timestamp = pickup_timestamp) AS zero_duration_rows,
    COUNT(*) FILTER (WHERE dropoff_timestamp > pickup_timestamp AND date_diff('minute', pickup_timestamp, dropoff_timestamp) > 1440) AS over_24h_rows
FROM taxi_all
""").fetchdf()
date_summary.to_csv(interim_dir / "audit_date_summary.csv", index=False)
date_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,pickup_before_window,pickup_after_window,dropoff_before_window,dropoff_after_allowed_spillover,corrupt_timestamp_rows,inverted_rows,zero_duration_rows,over_24h_rows
0,48601782,12,2,9,1,8,1942,649668,404


The complete-corpus timestamp result is **8 corrupt rows**, **1,942 temporal inversions**, **649,668 zero-duration rows**, and **404 trips over 24 hours**. The earlier count of 14 corrupt records is not used because the canonical rule explicitly checks both timestamp fields and returns 8 rows.

## 6. Zone reference integrity

The 265-row reference contains IDs 1 through 265, including `264 = Unknown`, `265 = Outside of NYC`, `132 = JFK Airport`, and `138 = LaGuardia Airport`. Endpoint joins are checked separately for origin and destination.

In [9]:
zone_summary = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE origin.loc_id IS NULL) AS unmapped_origin_rows,
    COUNT(*) FILTER (WHERE destination.loc_id IS NULL) AS unmapped_destination_rows,
    COUNT(*) FILTER (WHERE t.origin_loc_id IS NULL) AS null_origin_rows,
    COUNT(*) FILTER (WHERE t.dest_loc_id IS NULL) AS null_destination_rows,
    COUNT(*) FILTER (WHERE t.origin_loc_id IN (264, 265)) AS unknown_or_outside_origin_rows,
    COUNT(*) FILTER (WHERE t.dest_loc_id IN (264, 265)) AS unknown_or_outside_destination_rows,
    COUNT(*) FILTER (WHERE t.origin_loc_id IN (264, 265) AND t.dest_loc_id IN (264, 265)) AS unknown_or_outside_both_rows
FROM taxi_all t
LEFT JOIN zones origin ON t.origin_loc_id = origin.loc_id
LEFT JOIN zones destination ON t.dest_loc_id = destination.loc_id
""").fetchdf()
zone_summary.to_csv(interim_dir / "audit_zone_summary.csv", index=False)
zone_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unmapped_origin_rows,unmapped_destination_rows,null_origin_rows,null_destination_rows,unknown_or_outside_origin_rows,unknown_or_outside_destination_rows,unknown_or_outside_both_rows
0,48601782,0,0,0,0,92853,317258,49230


All taxi endpoints map to the reference: **0 unmapped origins** and **0 unmapped destinations**. There are **92,853 unknown/outside origins**, **317,258 unknown/outside destinations**, and **49,230 rows where both endpoints are special IDs**. These rows remain available for demand counts but will be flagged for spatial interpretation.

## 7. Canonical cleaning rules

The flags below intentionally overlap. The two final dispositions are deduplicated unions:

- `full_clean` drops records that are unusable for timestamps, duration, speed, or distance.
- `model_clean` additionally excludes negative fare/charge records and standard-rate zero-distance capture errors.
- Negative values in other monetary fields are profiled but remain diagnostic until their business meaning is established.
- Fare reconciliation is a flag only, using a `$0.05` tolerance.

In [10]:
rule_sql = f"""
WITH lineage AS (
    SELECT *, row_number() OVER (PARTITION BY {column_sql} ORDER BY source_file, source_row_number) AS duplicate_rank
    FROM taxi_all
), flags AS (
    SELECT *,
        duplicate_rank > 1 AS is_exact_duplicate,
        pickup_timestamp < TIMESTAMP '2020-01-01' OR dropoff_timestamp < TIMESTAMP '2020-01-01' AS is_corrupt_timestamp,
        dropoff_timestamp < pickup_timestamp AS is_inverted_timestamp,
        dropoff_timestamp = pickup_timestamp AS is_zero_duration,
        dropoff_timestamp > pickup_timestamp AND date_diff('minute', pickup_timestamp, dropoff_timestamp) > 1440 AS is_over_24h,
        dropoff_timestamp > pickup_timestamp AND distance_miles > 0 AND distance_miles / (date_diff('second', pickup_timestamp, dropoff_timestamp) / 3600.0) > 100 AS is_speed_over_100,
        dropoff_timestamp > pickup_timestamp AND distance_miles > 0 AND distance_miles / (date_diff('second', pickup_timestamp, dropoff_timestamp) / 3600.0) < 1 AND date_diff('minute', pickup_timestamp, dropoff_timestamp) > 60 AS is_stuck_meter,
        distance_miles > 100 AS is_extreme_distance,
        base_fare < 0 OR charge_total < 0 AS is_negative_fare,
        distance_miles = 0 AND (base_fare != 0 OR charge_total != 0) AS is_zero_distance_nonzero_fare,
        rate_class_id = 1 AND distance_miles = 0 AND (base_fare != 0 OR charge_total != 0) AND NOT (base_fare < 0 OR charge_total < 0) AS is_standard_zero_distance_drop,
        rider_count IS NULL AS is_null_rider_count,
        rider_count = 0 AS is_zero_rider_count,
        charge_total IS NOT NULL AND ABS(charge_total - (COALESCE(base_fare, 0) + COALESCE(surcharge_misc, 0) + COALESCE(transit_tax, 0) + COALESCE(driver_tip_payment, 0) + COALESCE(toll_total, 0) + COALESCE(service_improvement_fee, 0) + COALESCE(zone_congestion_fee, 0) + COALESCE(Airport_fee, 0) + COALESCE(congestion_relief_fee, 0))) > 0.05 AS fare_reconciliation_mismatch
    FROM lineage
), dispositions AS (
    SELECT *,
        is_exact_duplicate OR is_corrupt_timestamp OR is_inverted_timestamp OR is_zero_duration OR is_over_24h OR is_speed_over_100 OR is_stuck_meter OR is_extreme_distance AS drop_from_full_clean,
        is_negative_fare OR is_standard_zero_distance_drop AS drop_from_model_clean
    FROM flags
)
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE drop_from_full_clean) AS drop_union_full_clean,
    COUNT(*) FILTER (WHERE NOT drop_from_full_clean) AS retain_full_clean,
    COUNT(*) FILTER (WHERE drop_from_full_clean OR drop_from_model_clean) AS drop_union_model_clean,
    COUNT(*) FILTER (WHERE NOT (drop_from_full_clean OR drop_from_model_clean)) AS retain_model_clean,
    COUNT(*) FILTER (WHERE is_negative_fare) AS negative_fare_or_charge,
    COUNT(*) FILTER (WHERE is_zero_distance_nonzero_fare) AS zero_distance_nonzero_fare,
    COUNT(*) FILTER (WHERE is_standard_zero_distance_drop) AS standard_zero_distance_nonnegative,
    COUNT(*) FILTER (WHERE is_stuck_meter) AS stuck_meter_over_60_min_under_1_mph,
    COUNT(*) FILTER (WHERE fare_reconciliation_mismatch) AS fare_reconciliation_mismatch_over_005
FROM dispositions
"""
rule_summary = con.execute(rule_sql).fetchdf()
rule_summary.to_csv(interim_dir / "audit_rule_summary.csv", index=False)
rule_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,drop_union_full_clean,retain_full_clean,drop_union_model_clean,retain_model_clean,negative_fare_or_charge,zero_distance_nonzero_fare,standard_zero_distance_nonnegative,stuck_meter_over_60_min_under_1_mph,fare_reconciliation_mismatch_over_005
0,48601782,683399,47918383,3299329,44459194,2405603,1476006,223649,19204,17386879


## 8. Verified audit findings

The following values were independently verified by executing the canonical audit against all 48,601,782 raw records. They are written here as results, not assumptions, and should replace the superseded figures in the implementation plan.

### Corpus and lineage

- Complete taxi corpus: **48,601,782 rows**.
- Monthly files: **12**.
- Zone reference: **265 rows**, IDs 1 through 265.
- Exact duplicate rows to remove: **1**.

### Timestamp and movement quality

- Corrupt pre-2020 timestamp rows: **8**.
- Pickup timestamps before the nominal window: **12**.
- Dropoff timestamps before the nominal window: **9**.
- Temporal inversions: **1,942**.
- Zero-duration rows: **649,668**.
- Trips over 24 hours: **404**.
- Speed over 100 mph: **11,899**.
- Stuck-meter rule, speed under 1 mph for over 60 minutes: **19,204**.
- Distance over 100 miles: **2,817**.

### Classification and disposition

- Negative `base_fare` or `charge_total`: **2,405,603**, retained as flagged records in `full_clean`, excluded from `model_clean`.
- Zero-distance/nonzero-fare records: **1,476,006**, retained for classification rather than blanket deletion.
- Standard-rate, nonnegative zero-distance records: **223,649**, excluded from `model_clean`.
- Null rider counts: **12,405,268**, imputed to 1 with flags.
- Exact-zero rider counts: **231,578**, imputed to 1 with flags.
- Fare reconciliation mismatches over $0.05: **17,386,879**, diagnostic flag only.
- Final `full_clean` drop union: **683,399** rows; retained: **47,918,383**.
- Final `model_clean` drop union: **3,299,329** rows; retained: **44,459,194**.

These counts are intentionally not calculated by adding anomaly rows because the rules overlap.

## 9. Pre-cleaning contract

Cleaning may begin only after this notebook completes without assertion failures. The cleaner must preserve lineage and flags, rename `Airport_fee` to `airport_pickup_fee`, impute rider counts to 1, retain flagged business segments in `full_clean`, and apply the deduplicated unions above.

The chronological split must use pickup month: April 2025 through January 2026 for training, February 2026 for validation, and March 2026 for testing. The split sizes must be recomputed after cleaning; the old planned counts are superseded.

In [11]:
expected = {
    "total_rows": 48_601_782,
    "drop_union_full_clean": 683_399,
    "retain_full_clean": 47_918_383,
    "drop_union_model_clean": 3_299_329,
    "retain_model_clean": 44_459_194,
    "negative_fare_or_charge": 2_405_603,
    "zero_distance_nonzero_fare": 1_476_006,
    "standard_zero_distance_nonnegative": 223_649,
    "stuck_meter_over_60_min_under_1_mph": 19_204,
    "fare_reconciliation_mismatch_over_005": 17_386_879,
}
verified = rule_summary.iloc[0].to_dict()
for field, value in expected.items():
    assert int(verified[field]) == value, f"{field}: expected {value}, got {verified[field]}"

assert int(date_summary.loc[0, "corrupt_timestamp_rows"]) == 8
assert int(date_summary.loc[0, "inverted_rows"]) == 1_942
assert int(date_summary.loc[0, "zero_duration_rows"]) == 649_668
assert int(zone_summary.loc[0, "unmapped_origin_rows"]) == 0
assert int(zone_summary.loc[0, "unmapped_destination_rows"]) == 0
print("Canonical audit assertions passed.")

Canonical audit assertions passed.
